# 20 · Item-Based Collaborative Filtering (Covisitation kNN)

**Purpose.** First genuinely personalized *discovery* model: recommend items
that frequently co-occur in baskets with what the household recently bought.
This is the classic item-to-item CF pattern (Amazon, 2003) — the answer to
the baseline notebook's finding that catalog partitions (departments) don't
personalize, but behavioral similarity might.

## Theory

**Similarity from co-occurrence.** For products *a, b*, count the baskets
containing both. Two scores derive from the counts:

- `pair_baskets` — raw co-basket count; favors associations with popular items
- `lift = P(a,b) / (P(a)·P(b))` — co-occurrence relative to what independence
  predicts; >1 means the pair travels together more than chance. Favors
  tight-but-niche associations.

**Scoring a household.** Take the household's recent items (last 90 days) as
the query set H, then for every neighbor item *i*:

    score(household, i) = Σ over j ∈ H of sim(j, i)

Rank by score, keep top K.

**Leakage control.** Similarities come from `covisitation_pairs(as_of)` — a
macro computing co-occurrence *only from baskets on or before the snapshot
day* (the static full-period mart is EDA-only). Min support: 5 shared baskets.

**Evaluation.** Protocol identical to notebook 10 (as-of 600, 30-day labels,
@10 metrics, oracle ceiling 0.402) — plus a **discovery-slice** evaluation:
labels restricted to products the household never bought before as-of, where
buy-again scores zero by construction. That slice is CF's fair test; the
total-recall leaderboard is structurally rigged toward repeat purchases.

In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path

from retail_ds.evaluate.metrics import recall_at_k, hit_rate_at_k, ndcg_at_k

ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parents[1]

CFG = yaml.safe_load((ROOT / "configs" / "base.yaml").read_text())
con = duckdb.connect((ROOT / "db" / "retail.duckdb").as_posix(), read_only=True)

def q(sql: str) -> pd.DataFrame:
    return con.sql(sql).df()

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 4.5)

AS_OF = CFG["snapshots"]["valid"]   # 600
K = CFG["top_k"]                    # 10
labels = q(f"SELECT household_key, product_id FROM purchase_labels({AS_OF}, 30)")
rows = []                           # this notebook's leaderboard accumulator

print("as_of:", AS_OF, "| households with labels:", labels["household_key"].nunique())

as_of: 600 | households with labels: 2054


In [2]:
# Variant 0 — raw lift, no neighbor cap (kept as the instructive failure)
item_knn = q(f"""
    WITH history AS (
        SELECT household_key, product_id
        FROM household_product_snapshot({AS_OF})
        WHERE days_since_last <= 90
    ),
    scores AS (
        SELECT h.household_key, n.neighbor AS product_id, SUM(n.lift) AS score
        FROM history h
        JOIN covisitation_pairs({AS_OF}) n ON h.product_id = n.source
        GROUP BY h.household_key, n.neighbor
    )
    SELECT household_key, product_id,
           ROW_NUMBER() OVER (PARTITION BY household_key ORDER BY score DESC) AS rank
    FROM scores
    QUALIFY rank <= {K}
""")

rows.append({"model": "item_knn_lift",
             f"recall@{K}":   recall_at_k(item_knn, labels, K),
             f"hit_rate@{K}": hit_rate_at_k(item_knn, labels, K),
             f"ndcg@{K}":     ndcg_at_k(item_knn, labels, K)})
pd.DataFrame(rows).set_index("model").round(3)

,recall@10,hit_rate@10,ndcg@10
model,,,
item_knn_lift,0.012,0.242,0.058


**Findings — raw lift kNN.** recall@10 = 0.012 · hit_rate@10 = 0.242 — loses
even to global popularity (0.041 / 0.703). Mechanism: lift's small-denominator
explosion. For two items each in ~10 baskets co-occurring in 5, lift ≈ 13,500;
for bananas+milk, genuinely predictive, lift ≈ 2. Summing raw lift lets
ultra-rare pairs with near-zero base rates hijack the ranking, and with no
per-item neighbor cap the noise accumulates without bound. Lift measures
*surprise*, not *usefulness*.

**Fix (standard item-kNN hygiene):** damp association by evidence
(`lift · ln(1 + pair_baskets)`) and cap neighbors at top-50 per source item.

In [3]:
def knn_recs(score_expr: str, top_neighbors: int = 50) -> pd.DataFrame:
    return q(f"""
        WITH history AS (
            SELECT household_key, product_id
            FROM household_product_snapshot({AS_OF})
            WHERE days_since_last <= 90
        ),
        neighbors AS (
            SELECT source, neighbor, pair_baskets, lift,
                   ROW_NUMBER() OVER (
                       PARTITION BY source ORDER BY {score_expr} DESC
                   ) AS nb_rank
            FROM covisitation_pairs({AS_OF})
        ),
        scores AS (
            SELECT h.household_key, n.neighbor AS product_id,
                   SUM({score_expr}) AS score
            FROM history h
            JOIN neighbors n ON h.product_id = n.source
            WHERE n.nb_rank <= {top_neighbors}
            GROUP BY h.household_key, n.neighbor
        )
        SELECT household_key, product_id,
               ROW_NUMBER() OVER (PARTITION BY household_key ORDER BY score DESC) AS rank
        FROM scores
        QUALIFY rank <= {K}
    """)

variants = {}
for name, expr in {
    "item_knn_counts":      "pair_baskets",
    "item_knn_damped_lift": "lift * ln(1 + pair_baskets)",
}.items():
    variants[name] = knn_recs(expr)
    rows.append({"model": name,
                 f"recall@{K}":   recall_at_k(variants[name], labels, K),
                 f"hit_rate@{K}": hit_rate_at_k(variants[name], labels, K),
                 f"ndcg@{K}":     ndcg_at_k(variants[name], labels, K)})

In [4]:
oracle = (labels.groupby("household_key").size().clip(upper=K)
          / labels.groupby("household_key").size()).mean()

knn_board = pd.DataFrame(rows).set_index("model").round(3)
knn_board["share_of_ceiling"] = (knn_board[f"recall@{K}"] / oracle).round(2)

baselines = pd.read_csv(ROOT / "reports" / "baseline_leaderboard.csv", index_col="model")
full_board = pd.concat([baselines, knn_board]).sort_values(f"recall@{K}", ascending=False)
full_board.to_csv(ROOT / "reports" / "leaderboard.csv")
full_board

,recall@10,hit_rate@10,ndcg@10,share_of_ceiling
model,,,,
buy_again,0.072,0.757,0.303,0.18
global_popularity,0.041,0.703,0.192,0.10
dept_popularity,0.034,0.650,0.155,0.08
item_knn_counts,0.033,0.650,0.175,0.08
item_knn_damped_lift,0.013,0.247,0.064,0.03
item_knn_lift,0.012,0.242,0.058,0.03


In [5]:
# global popularity, recomputed here (notebook 10's variables live in its own kernel)
global_pop = q(f"""
    WITH top_products AS (
        SELECT product_id, COUNT(*) AS n
        FROM staging.stg_transactions
        WHERE day_no <= {AS_OF} AND day_no > {AS_OF} - 365
        GROUP BY product_id
        ORDER BY n DESC LIMIT {K}
    ),
    households AS (
        SELECT DISTINCT household_key FROM customer_snapshot({AS_OF})
    )
    SELECT h.household_key, t.product_id,
           ROW_NUMBER() OVER (PARTITION BY h.household_key ORDER BY t.n DESC) AS rank
    FROM households h CROSS JOIN top_products t
""")

# discovery labels: bought in the window, never bought on or before as_of
labels_new = q(f"""
    SELECT l.household_key, l.product_id
    FROM purchase_labels({AS_OF}, 30) l
    LEFT JOIN household_product_snapshot({AS_OF}) h
      USING (household_key, product_id)
    WHERE h.product_id IS NULL
""")
print(f"discovery pairs: {len(labels_new):,} = {len(labels_new)/len(labels):.1%} of all label pairs")

discovery pairs: 56,710 = 53.2% of all label pairs


In [6]:
# discovery-slice evaluation: buy-again scores 0 here by construction
disc_rows = []
for name, recs in {"global_popularity": global_pop, **variants}.items():
    disc_rows.append({"model": name,
                      f"disc_recall@{K}":   recall_at_k(recs, labels_new, K),
                      f"disc_hit_rate@{K}": hit_rate_at_k(recs, labels_new, K)})
pd.DataFrame(disc_rows).set_index("model").round(3)

,disc_recall@10,disc_hit_rate@10
model,,
global_popularity,0.007,0.094
item_knn_counts,0.005,0.088
item_knn_damped_lift,0.003,0.055


**Findings — item-kNN (as-of 600, K=10). FILL AFTER RUN ALL.**

Total-recall board: counts ≫ damped lift ≫ raw lift (0.033 / 0.013 / 0.012) —
the co-occurrence signal here is mostly co-popularity; association-strength
weighting adds nothing. On total recall kNN cannot beat buy-again, by
construction: labels are ~55% repeats, which buy-again owns.

Discovery slice ({FILL}% of label pairs, where buy-again scores 0 by
construction): {FILL — kNN counts vs global popularity: winner and numbers}.

**Decisions:** {FILL — covisitation's candidate-generator role confirmed or
rejected, with its config: counts scoring, top-50 neighbors, 90d history}.
Solo top-10 leaderboards understate candidate sources — from here, generators
are judged on slice-specific and pooled candidate recall (notebook 30),
models on ranked top-K.